# Project 3 — RAG Monitoring & Observability

Wraps a RAG pipeline with **tracing + quality metrics + regression gating**. Self-contained (builds its own mini-RAG). Needs `OPENAI_API_KEY` in `../.env`.

Langfuse is **optional** — if `LANGFUSE_*` keys are set it traces there; otherwise it traces to an in-notebook store. Either way it computes P50/P95 latency, cost/request, citation coverage, failure rate, and emits **`RESULTS.md`**.

Self-host Langfuse (optional): `git clone https://github.com/langfuse/langfuse && cd langfuse && docker compose up`.

In [ ]:
%pip install -q langchain langchain-community langchain-openai chromadb \
    sentence-transformers langfuse python-dotenv pandas numpy

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('OPENAI_API_KEY'), 'Put OPENAI_API_KEY in ../.env'

LANGFUSE_ON = bool(os.getenv('LANGFUSE_PUBLIC_KEY') and os.getenv('LANGFUSE_SECRET_KEY'))
lf = None
if LANGFUSE_ON:
    from langfuse import Langfuse
    lf = Langfuse()
    print('Langfuse tracing ON ->', os.getenv('LANGFUSE_HOST'))
else:
    print('Langfuse keys absent -> using in-notebook trace store')

### Mini-RAG under observation
Same Acme Cloud corpus as Project 1 so the eval set lines up.

In [ ]:
import textwrap
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma

CORPUS = {
  'pricing': 'Acme Cloud Free tier: 1 project, 500 MB storage, 10,000 API calls/month, free. Pro tier: $29/month, 10 projects, 50 GB storage, 2 million API calls/month. Enterprise: custom price, unlimited projects, 99.99% uptime SLA, SSO.',
  'auth': 'Acme Cloud supports API key auth (Bearer token in Authorization header) and OAuth 2.0. API keys rotate from Settings; rotated keys valid 24-hour grace. OAuth access tokens expire after 1 hour, refresh tokens after 30 days. SSO via SAML with Okta and Azure AD.',
  'limits': 'Default rate limit is 100 requests/second per API key; exceeding returns HTTP 429 with Retry-After. Pro can request up to 500 rps. Batch endpoints accept up to 1,000 items. Webhooks retried up to 5 times with exponential backoff.',
  'regions': 'Acme Cloud regions: us-east-1, us-west-2, eu-central-1, ap-southeast-1. eu-central-1 is GDPR-compliant in Frankfurt. Cross-region replication is Enterprise-only. Default region is us-east-1.'
}
docs = [Document(page_content=v, metadata={'source': k}) for k, v in CORPUS.items()]
chunks = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100).split_documents(docs)
emb = OpenAIEmbeddings(model='text-embedding-3-small')
vs = Chroma.from_documents(chunks, emb, collection_name='obs', persist_directory='./chroma_obs')
retriever = vs.as_retriever(search_kwargs={'k': 3})
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('mini-RAG ready')

## Phase 1 — Instrument every step (tracing)
Capture retrieved chunks, prompt, response, token usage, and per-step latency for every request.

In [ ]:
import time
# Approx OpenAI pricing (USD per 1M tokens) for cost/request accounting
PRICE_IN, PRICE_OUT = 0.15, 0.60  # gpt-4o-mini
PROMPT = ('Answer ONLY from context, cite chunk numbers like [1]. '
          'If unsupported, reply exactly: I cannot answer this from the documents.\n\n'
          'Context:\n{ctx}\n\nQuestion: {q}')
TRACES = []

def traced_rag(question):
    t0 = time.perf_counter()
    hits = retriever.invoke(question)
    t_ret = time.perf_counter()
    ctx = '\n\n'.join(f'[{i+1}] {d.page_content}' for i, d in enumerate(hits))
    msg = PROMPT.format(ctx=ctx, q=question)
    error = None
    try:
        resp = llm.invoke(msg)
        answer = resp.content
        usage = resp.response_metadata.get('token_usage', {})
    except Exception as e:
        answer, usage, error = '', {}, str(e)
    t_gen = time.perf_counter()
    tin = usage.get('prompt_tokens', 0); tout = usage.get('completion_tokens', 0)
    cost = (tin * PRICE_IN + tout * PRICE_OUT) / 1e6
    cited = any(f'[{i+1}]' in answer for i in range(len(hits)))
    declined = 'cannot answer' in answer.lower()
    trace = {'question': question, 'answer': answer,
             'retrieval_ms': round((t_ret - t0) * 1000, 1),
             'generation_ms': round((t_gen - t_ret) * 1000, 1),
             'total_ms': round((t_gen - t0) * 1000, 1),
             'tokens_in': tin, 'tokens_out': tout, 'cost_usd': cost,
             'cited': cited, 'declined': declined, 'error': error,
             'sources': [d.metadata['source'] for d in hits]}
    TRACES.append(trace)
    if lf:
        lf.trace(name='rag', input=question, output=answer,
                 metadata={k: trace[k] for k in ('total_ms', 'cost_usd', 'cited')})
    return trace

print(traced_rag('How much does the Pro tier cost?'))

## Phase 2 — Quality metrics over time
Run the workload, then compute P50/P95 latency, cost/request, citation coverage, failure rate.

In [ ]:
WORKLOAD = [
  'How much does the Pro tier cost?', 'How many API calls does the Free tier include?',
  'How are API keys passed?', 'When do OAuth access tokens expire?',
  'What is the default rate limit?', 'What status code is returned when rate limited?',
  'Which regions does Acme Cloud operate in?', 'Where is the GDPR region located?',
  'Which tier offers cross-region replication?', 'What is the default region?',
  'What uptime SLA does Enterprise offer?', 'How many times are webhooks retried?',
  'What is the capital of France?',  # out-of-domain -> should be declined
  'What is the meaning of life?',    # out-of-domain -> should be declined
]
TRACES.clear()
for q in WORKLOAD:
    traced_rag(q)
print(len(TRACES), 'requests traced')

In [ ]:
import numpy as np, pandas as pd
df = pd.DataFrame(TRACES)
lat = df['total_ms'].to_numpy()
in_domain = df[~df['declined']]
metrics = {
  'requests': len(df),
  'latency_p50_ms': round(float(np.percentile(lat, 50)), 1),
  'latency_p95_ms': round(float(np.percentile(lat, 95)), 1),
  'latency_mean_ms': round(float(lat.mean()), 1),
  'cost_per_request_usd': round(float(df['cost_usd'].mean()), 6),
  'citation_coverage_pct': round(100 * in_domain['cited'].mean(), 1) if len(in_domain) else 0.0,
  'failure_rate_pct': round(100 * df['error'].notna().mean(), 1),
  'declined_pct': round(100 * df['declined'].mean(), 1),
}
metrics

## Phase 3 — Regression gating
Define thresholds. If any key metric breaches, the build fails (blocks merge in CI).

In [ ]:
GATES = {
  'latency_p95_ms': ('<=', 8000),
  'citation_coverage_pct': ('>=', 90),
  'failure_rate_pct': ('<=', 5),
}
import operator
OPS = {'<=': operator.le, '>=': operator.ge}
results = {k: OPS[op](metrics[k], thr) for k, (op, thr) in GATES.items()}
passed = all(results.values())
for k, (op, thr) in GATES.items():
    print(f'{k} {metrics[k]} {op} {thr} -> {"PASS" if results[k] else "FAIL"}')
print('OVERALL:', 'PASS' if passed else 'FAIL')

In [ ]:
# Write RESULTS.md
md = ['# Project 3 — RAG Observability Report', '',
      f'Traced {metrics["requests"]} requests. Langfuse: {"ON" if lf else "in-notebook store"}.',
      '', '## Quality Metrics', '', '| Metric | Value |', '|---|---|']
for k, v in metrics.items():
    md.append(f'| {k} | {v} |')
md += ['', '## Regression Gates', '', '| Gate | Condition | Result |', '|---|---|---|']
for k, (op, thr) in GATES.items():
    md.append(f'| {k} | {op} {thr} | {"PASS ✅" if results[k] else "FAIL ❌"} |')
md += ['', f'**Overall: {"PASS ✅" if passed else "FAIL ❌"}**', '',
       '## Per-request trace sample', '', df[['question','total_ms','cost_usd','cited','declined']].head(8).to_markdown(index=False)]
with open('RESULTS.md', 'w') as f:
    f.write('\n'.join(md))
print('Wrote RESULTS.md')
if lf: lf.flush()
assert passed, 'REGRESSION GATE FAILED — would block merge'
print('Gate passed')